### Part A: Divide our documents into chunks

In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go



c:\Users\thiru\Desktop\Engineering\AI Engineering\Projects\Course Projects\ai engineering\RAG Experiments\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print("Key is available ")
else:
    print("Key is not available")
    

Key is available 


In [3]:
knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base.")

entire_knowledge_base = ""

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        content = f.read()
        entire_knowledge_base += content + "\n"
print(f"Total characters in the knowledge base: {len(entire_knowledge_base)}")


Found 76 files in the knowledge base.
Total characters in the knowledge base: 304358


In [4]:
encoding = tiktoken.encoding_for_model(MODEL)
num_tokens = len(encoding.encode(entire_knowledge_base))
print(f"Total tokens in the knowledge base: {num_tokens}")

Total tokens in the knowledge base: 63549


In [5]:
folders = glob.glob("knowledge-base/*")
documents = []

for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", show_progress=True, silent_errors=True, loader_kwargs={"encoding": "utf-8"})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)
print(f"Total documents loaded: {len(documents)}")

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:02<00:00,  3.47it/s]

Total documents loaded: 76


In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(chunks)}")
print(f"first chunk: {chunks[0]}")

Total chunks created: 371
first chunk: page_content='About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.' metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}


In [20]:
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", openai_api_key=openai_api_key)
if os.path.exists(db_name):
    print("Loading existing vector database...")
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vector_store = Chroma.from_documents(chunks, embeddings, persist_directory=db_name)
print(f"Vector store created with {vector_store._collection.count()} vectors.")

Loading existing vector database...
Vector store created with 371 vectors.


In [21]:
collection = vector_store._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"there are {count} vectors in the collection, each with {dimensions} dimensions.")

there are 371 vectors in the collection, each with 3072 dimensions.


In [22]:
result = collection.get(include=["metadatas", "embeddings", "documents"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [meta["doc_type"] for meta in metadatas]
colors = [["blue", "red", "green", "orange"][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [23]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(color=colors, size=10, opacity=0.7),
    text=[f"{meta['doc_type']}: {doc[:50]}..." for meta, doc in zip(metadatas, documents)],
    hoverinfo='text'
))
fig.update_layout(title="t-SNE Visualization of Document Embeddings", xaxis_title="Dimension 1", yaxis_title="Dimension 2")
fig.show()

In [24]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)
# Use Scatter3d instead of Scatter
fig = go.Figure(data=go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(color=colors, size=5, opacity=0.7), # size 10 might be too big in 3D, adjusted to 5
    text=[f"{meta['doc_type']}: {doc[:50]}..." for meta, doc in zip(metadatas, documents)],
    hoverinfo='text'
))

# 3D axis titles must go inside the "scene" layout property
fig.update_layout(
    title="t-SNE Visualization of Document Embeddings", 
    scene=dict(
        xaxis_title="Dimension 1", 
        yaxis_title="Dimension 2",
        zaxis_title="Dimension 3"
    )
)
fig.show()